# PointNet (ShapeNetPart) — Runpod Notebook
Self-contained workflow for the bonus segmentation track: convert the raw ShapeNetPart annotations, clone the official PointNet repo, patch it for Runpod, train/fine-tune `train_partseg.py`, evaluate mIoU, and export artifacts.


In [ ]:
#@title 0) System info
import os
import platform
import subprocess
import sys
from datetime import datetime

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA in torch:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU count:', torch.cuda.device_count())
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"  - cuda:{idx} -> {props.name} ({props.total_memory/1e9:.1f} GB)")
except ImportError:
    print('PyTorch not installed; install it before running training cells.')
print('Working dir:', os.getcwd())
print('Timestamp:', datetime.now())


In [ ]:
#@title 1) Config — paths & hyper-parameters
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

RAW_BASE = NOTEBOOK_DIR / 'PartAnnotation'
OUT_ROOT = NOTEBOOK_DIR / 'shapenetcore_partanno_segmentation_benchmark_v0_normal'
REPO_URL = 'https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git'
REPO_BRANCH = 'master'
REPO_SUBDIR = 'Pointnet_Pointnet2_pytorch'
EXP_NAME = 'pointnet_partseg_xyz_runpod'
NUM_POINTS = 2048
BATCH_SIZE = 16
EPOCHS = 250
FINE_TUNE_EPOCHS = 320
LEARNING_RATE = 0.001
FINE_TUNE_LR = 3e-4
NUM_WORKERS = 12
PIN_MEMORY = True
AUTO_FETCH_DATASET = False
INSTALL_REQUIREMENTS = True
APPLY_RUNPOD_PATCH = True

REPO_DIR = (NOTEBOOK_DIR / REPO_SUBDIR).resolve()
DATA_PATH = OUT_ROOT
LOG_BASE = REPO_DIR / 'log' / 'part_seg' / EXP_NAME
LOG_BASE.mkdir(parents=True, exist_ok=True)
print('Notebook dir :', NOTEBOOK_DIR)
print('Raw ShapeNet :', RAW_BASE)
print('Processed out :', OUT_ROOT)
print('Repo dir      :', REPO_DIR)
print('Log dir       :', LOG_BASE)


In [ ]:
#@title 2) Verify raw dataset availability
if not RAW_BASE.exists():
    raise FileNotFoundError(f'RAW_BASE missing at {RAW_BASE}. Please place the ShapeNetPart PartAnnotation tree there.')
print('Found RAW dataset under', RAW_BASE)


In [ ]:
#@title 3) Convert raw PartAnnotation → PointNet txt format
import glob
import json
import numpy as np
import os
import shutil

def extract_uuid(path_str):
    base = os.path.basename(path_str)
    name, _ = os.path.splitext(base)
    return name

def load_list(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return {extract_uuid(entry) for entry in data}

def ensure_split_files(raw_base, out_root):
    split_dir = out_root / 'train_test_split'
    if split_dir.exists():
        return
    cand = list(raw_base.glob('**/train_test_split'))
    if not cand:
        raise RuntimeError('train_test_split folder not found under RAW_BASE')
    split_dir.mkdir(parents=True, exist_ok=True)
    for name in ['shuffled_train_file_list.json','shuffled_val_file_list.json','shuffled_test_file_list.json']:
        src = cand[0] / name
        if not src.exists():
            raise RuntimeError(f'Missing {src}')
        shutil.copy2(src, split_dir / name)
    synmap = cand[0].parent / 'synsetoffset2category.txt'
    if synmap.exists():
        shutil.copy2(synmap, out_root / 'synsetoffset2category.txt')

def convert_raw_to_txt(raw_base, out_root):
    ensure_split_files(raw_base, out_root)
    split_dir = out_root / 'train_test_split'
    train_ids = load_list(split_dir / 'shuffled_train_file_list.json')
    val_ids = load_list(split_dir / 'shuffled_val_file_list.json')
    test_ids = load_list(split_dir / 'shuffled_test_file_list.json')
    allowed = train_ids | val_ids | test_ids
    wrote = 0
    for syn_dir in sorted(raw_base.iterdir()):
        if not syn_dir.is_dir():
            continue
        for uuid_dir in syn_dir.iterdir():
            if not uuid_dir.is_dir():
                continue
            uuid = uuid_dir.name
            if uuid not in allowed:
                continue
            pts_candidates = list((uuid_dir / 'points').glob('*.pts')) if (uuid_dir / 'points').exists() else list(uuid_dir.glob('*.pts'))
            seg_candidates = list((uuid_dir / 'points_label').glob('*.seg'))
            if (uuid_dir / 'seg').exists():
                seg_candidates += list((uuid_dir / 'seg').glob('*.seg'))
            seg_candidates += list(uuid_dir.glob('*.seg'))
            if not pts_candidates or not seg_candidates:
                continue
            pts = np.loadtxt(pts_candidates[0]).astype(np.float32)
            if pts.ndim == 1:
                pts = pts.reshape(-1, pts.shape[0])
            seg = np.loadtxt(seg_candidates[0]).astype(np.int32).reshape(-1)
            n = min(len(pts), len(seg))
            pts = pts[:n, :3]
            seg = seg[:n]
            zeros = np.zeros((n,3), dtype=np.float32)
            out = np.concatenate([pts, zeros, seg.reshape(-1,1)], axis=1)
            dst = out_root / syn_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            np.savetxt(dst / f'{uuid}.txt', out, fmt='%.6f %.6f %.6f %.6f %.6f %.6f %d')
            wrote += 1
    print('Converted samples:', wrote)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
existing = list(OUT_ROOT.glob('*/*.txt'))
if existing:
    print('Found', len(existing), '.txt files — skipping conversion.')
else:
    convert_raw_to_txt(RAW_BASE, OUT_ROOT)


In [ ]:
#@title 4) Clone/Pull official PointNet repo + install deps
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists at', REPO_DIR, '- assuming newest.')
if INSTALL_REQUIREMENTS:
    req = REPO_DIR / 'requirements.txt'
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=False)
    extras = ['h5py', 'scikit-learn', 'tqdm', 'matplotlib']
    subprocess.run([sys.executable, '-m', 'pip', 'install', *extras], check=False)
print('Repo ready at', REPO_DIR)


In [ ]:

#@title 5) Patch part-seg scripts for Runpod
from pathlib import Path

train_file = REPO_DIR / 'train_partseg.py'
test_file = REPO_DIR / 'test_partseg.py'


def add_backend_block(text):
    marker = 'torch.backends.cudnn.benchmark = True'
    if marker in text:
        return text
    insert_after = "from data_utils.ShapeNetDataLoader import PartNormalDataset
"
    block = (
        "
"
        "torch.backends.cudnn.benchmark = True
"
        "if hasattr(torch.backends.cuda, 'matmul') and hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
"
        "    torch.backends.cuda.matmul.allow_tf32 = True
"
        "if hasattr(torch, 'set_float32_matmul_precision'):
"
        "    torch.set_float32_matmul_precision('high')
"
    )
    return text.replace(insert_after, insert_after + block, 1)


def add_parser_args_train(text):
    snippet = (
        "    parser.add_argument('--num_workers', type=int, default=10, help='number of workers for dataloaders')
"
        "    parser.add_argument('--pin_memory', action='store_true', default=False, help='pin memory flag')
"
    )
    if '--pin_memory' in text:
        return text
    target = "    parser.add_argument('--lr_decay', type=float, default=0.5, help='decay rate for lr decay')

"
    return text.replace(target, target + snippet + "
", 1)


def add_parser_args_test(text):
    snippet = (
        "    parser.add_argument('--num_workers', type=int, default=10, help='number of workers for dataloaders')
"
        "    parser.add_argument('--pin_memory', action='store_true', default=False, help='pin memory flag')
"
    )
    if '--pin_memory' in text:
        return text
    target = "    parser.add_argument('--num_votes', type=int, default=3, help='aggregate segmentation scores with voting')
"
    return text.replace(target, target + snippet + "
", 1)

TRAIN_LOAD_OLD = "        checkpoint = torch.load(str(exp_dir) + '/checkpoints/best_model.pth')"
TRAIN_LOAD_NEW = "        checkpoint = torch.load(str(exp_dir) + '/checkpoints/best_model.pth', weights_only=False)"
TEST_LOAD_OLD = "    checkpoint = torch.load(str(experiment_dir) + '/checkpoints/best_model.pth')"
TEST_LOAD_NEW = "    checkpoint = torch.load(str(experiment_dir) + '/checkpoints/best_model.pth', weights_only=False)"


def replace_num_workers(text):
    text = text.replace('num_workers=10', 'num_workers=args.num_workers, pin_memory=args.pin_memory')
    text = text.replace('num_workers=4', 'num_workers=args.num_workers, pin_memory=args.pin_memory')
    return text


def apply_patches(path, replacements):
    text = path.read_text()
    original = text
    for func in replacements:
        text = func(text)
    if text != original:
        path.write_text(text)
        print('Patched', path.name)
    else:
        print('Already patched', path.name)

if APPLY_RUNPOD_PATCH:
    apply_patches(train_file, [
        add_backend_block,
        add_parser_args_train,
        replace_num_workers,
        lambda txt: txt.replace(TRAIN_LOAD_OLD, TRAIN_LOAD_NEW),
    ])
    apply_patches(test_file, [
        add_backend_block,
        add_parser_args_test,
        replace_num_workers,
        lambda txt: txt.replace(TEST_LOAD_OLD, TEST_LOAD_NEW),
    ])
else:
    print('APPLY_RUNPOD_PATCH is False — skipping patch step.')


In [ ]:
#@title 6) Link processed dataset into repo
import os
import shutil

repo_data = REPO_DIR / 'data'
repo_data.mkdir(exist_ok=True)
expected = repo_data / 'shapenetcore_partanno_segmentation_benchmark_v0_normal'
if expected.exists() or expected.is_symlink():
    if expected.is_symlink():
        expected.unlink()
    else:
        shutil.rmtree(expected)
try:
    expected.symlink_to(OUT_ROOT, target_is_directory=True)
    print('Symlinked dataset ->', expected)
except OSError:
    shutil.copytree(OUT_ROOT, expected)
    print('Copied dataset into repo data dir')


In [ ]:
#@title 7) Helper functions (training/testing)
import os
import shlex
import subprocess
import sys
from pathlib import Path

LOG_DIR = REPO_DIR / 'log' / 'part_seg' / EXP_NAME
LOG_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = LOG_DIR / 'checkpoints' / 'best_model.pth'


def run_cmd(cmd, log_file):
    print('Running:', ' '.join(shlex.quote(str(c)) for c in cmd))
    with open(log_file, 'a') as f:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
            f.write(line)
        proc.wait()
        if proc.returncode != 0:
            raise RuntimeError(f'Command failed with exit code {proc.returncode}')


def train_partseg(epochs, lr):
    cmd = [
        sys.executable,
        'train_partseg.py',
        '--log_dir', EXP_NAME,
        '--npoint', str(NUM_POINTS),
        '--batch_size', str(BATCH_SIZE),
        '--epoch', str(epochs),
        '--learning_rate', str(lr),
        '--num_workers', str(NUM_WORKERS),
    ]
    if PIN_MEMORY:
        cmd.append('--pin_memory')
    run_cmd(cmd, LOG_DIR / 'console_train.txt')


def eval_partseg():
    cmd = [
        sys.executable,
        'test_partseg.py',
        '--log_dir', EXP_NAME,
        '--npoint', str(NUM_POINTS),
        '--batch_size', str(BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
    ]
    if PIN_MEMORY:
        cmd.append('--pin_memory')
    run_cmd(cmd, LOG_DIR / 'console_eval.txt')


In [ ]:
#@title 8) Train baseline (PointNet PartSeg)
train_partseg(EPOCHS, LEARNING_RATE)


In [ ]:
#@title 9) Fine-tune baseline (lower LR)
train_partseg(FINE_TUNE_EPOCHS, FINE_TUNE_LR)


In [ ]:
#@title 10) Evaluate checkpoint only
if not CHECKPOINT.exists():
    raise FileNotFoundError('Checkpoint missing at ' + str(CHECKPOINT))
eval_partseg()


In [ ]:
#@title 11) Tail logs for report
from pathlib import Path

def tail(path, n=20):
    if not path.exists():
        return []
    with open(path) as f:
        lines = f.readlines()
    return lines[-n:]

print('=== Train tail ===')
print(''.join(tail(LOG_DIR / 'console_train.txt')))
print('=== Eval tail ===')
print(''.join(tail(LOG_DIR / 'console_eval.txt')))


In [ ]:
#@title 12) Export artifacts
import shutil
import time
from pathlib import Path

EXPORT_BASE = Path('pointnet_partseg_artifacts')
TARGET = EXPORT_BASE / EXP_NAME
if TARGET.exists():
    shutil.rmtree(TARGET)
shutil.copytree(LOG_DIR, TARGET / 'log')
(TARGET / 'NOTE.txt').write_text(
    'Exported at ' + time.ctime() + '
' +
    'Repo commit: ' + subprocess.getoutput(f"cd '{REPO_DIR}' && git rev-parse HEAD") + '
'
)
print('Exported to', TARGET)
